# Using Qwen3-0.6B in a Jupyter Notebook

This notebook shows a very small local LLM workflow with
[`Qwen/Qwen3-0.6B`](https://huggingface.co/Qwen/Qwen3-0.6B).

We will:

1. Load the model.
2. Ask normal questions.
3. Request and parse JSON.
4. Extract code from a response.
5. Turn a model response into a simple Python structure.

## 1. Install dependencies

In [ ]:
%pip install -U --force-reinstall torchvision==0.29.1

In [ ]:
%pip install -U "transformers>=4.51.0" accelerate torch

## 2. Load Qwen3-0.6B

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

MODEL_NAME = "Qwen/Qwen3-0.6B"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype="auto"
)

print("Model loaded on:", model.device)

## 3. A tiny helper function

`ask()` sends one user message to Qwen and returns only the generated answer.

`enable_thinking=False` keeps the examples short and makes structured output easier to inspect.

In [ ]:
def ask(prompt, max_new_tokens=200):
    messages = [
        {"role": "user", "content": prompt}
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        enable_thinking=False,
        tokenize=True,
        return_dict=True,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False
        )

    new_tokens = output[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        new_tokens,
        skip_special_tokens=True
    ).strip()

## 4. Normal text response

In [ ]:
answer = ask("Explain what courses Hof University has in two short sentences.")
print(answer)

## 5. Ask for JSON and parse it

JSON is useful when another program should consume the answer.

Important: the model still produces **text**.  
`json.loads()` converts valid JSON text into normal Python objects.

In [ ]:
import json

response = ask('''
Return ONLY valid JSON.

Create information about Python with exactly this structure:
{
  "language": "string",
  "easy_for_beginners": true,
  "uses": ["string", "string", "string"]
}
''')

print("Raw model output:")
print(response)

data = json.loads(response)

print("\nParsed Python object:")
print(data)

print("\nFirst use:")
print(data["uses"][0])

### A tiny helper for fenced JSON

Models sometimes wrap JSON in:

```text
```json
{ ... }
```
```

So we can clean it up writing a small helper function!

In [ ]:
def clean_code_fence(text):
    text = text.strip()

    if text.startswith("```"):
        lines = text.splitlines()
        lines = lines[1:]

        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]

        text = "\n".join(lines)

    return text.strip()


response = ask('''
Return ONLY JSON with these keys:
"name", "year", "active".

Use Ada Lovelace as the example.
''')

cleaned = clean_code_fence(response)
data = json.loads(cleaned)

print(data)

## 6. Ask the model for code and extract it

In [ ]:
response = ask('''
Write a Python function called square(x) that returns x squared.

Return the answer inside one ```python code block.
Do not add any explanation.
''')

print(response)

In [ ]:
import re

match = re.search(r"```python\s*(.*?)```", response, re.DOTALL)

if match:
    python_code = match.group(1).strip()
else:
    python_code = response.strip()

print("Extracted code:")
print(python_code)

### Optional: run the generated code

`exec()` executes Python code, so only do this with code you trust.

For this controlled lab example, we can run the tiny generated function.

In [ ]:
namespace = {}
exec(python_code, namespace)

print(namespace["square"](5))

## 7. Parse a simple classification result

For many tasks we do not need complicated JSON.

We can ask the model for one value from a small set of allowed labels.

In [ ]:
response = ask('''
Classify this message as exactly one of:
POSITIVE
NEGATIVE
NEUTRAL

Message:
"The lab was surprisingly easy to follow."

Return ONLY the label.
''')

label = response.strip().upper()

allowed_labels = {"POSITIVE", "NEGATIVE", "NEUTRAL"}

if label not in allowed_labels:
    print("Unexpected model output:", response)
else:
    print("Parsed label:", label)

## 8. Extract several fields at once

A common pattern is to turn unstructured text into structured information.

In [ ]:
text = '''
Alice is 24 years old.
She studies computer science in Munich.
Her favorite programming language is Python.
'''

response = ask(f'''
Extract information from the following text.

Return ONLY valid JSON using this structure:
{{
  "name": "string",
  "age": 0,
  "city": "string",
  "favorite_language": "string"
}}

Text:
{text}
''')

person = json.loads(clean_code_fence(response))

print(person)
print("Name:", person["name"])
print("Age:", person["age"])

## 9. Simple takeaway

The basic workflow is:

```python
text = ask("...")
```

Then decide what you expect:

- **Human-readable answer** → just print the text.
- **JSON** → `json.loads(...)`
- **Code block** → extract the fenced block.
- **Fixed label** → compare against allowed values.
- **Structured extraction** → ask for a predefined JSON schema.

For real applications, model output should always be validated because an LLM can still return unexpected text.